# Projet 3 — Modélisation du Churn

Ce notebook présente le pipeline de modélisation pour la prédiction du churn client.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

df = pd.read_csv(DATA_PATH)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.drop(columns=['customerID'], errors='ignore')

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

In [ ]:
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

model = LogisticRegression(max_iter=1000, solver='liblinear', class_weight='balanced', random_state=42)
pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])

In [ ]:
param_grid = {'model__C': [0.1, 1, 5, 10]}
search = GridSearchCV(pipe, param_grid=param_grid, scoring='roc_auc', cv=3, n_jobs=-1, refit=True)
search.fit(X_train, y_train)
print(search.best_params_)

In [ ]:
y_pred = search.predict(X_test)
y_proba = search.predict_proba(X_test)[:, 1]
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_proba),
}
print(metrics)
print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))

## Conclusion

Le modèle logistique choisi garde une bonne capacité de détection de churn sur le jeu de test, avec une précision acceptable et une ROC-AUC élevée. Une prochaine amélioration possible est l’ajout d’un modèle tree-based ou la sélection de variables plus ciblée.